In [1]:
import joblib
import numpy as np
import pandas as pd

import warnings

warnings.filterwarnings("ignore")

model_bin_path = "D:/Area de trabalho/Faculdade/BioData/Q-Cyber/QML/IA/weights/traditional/random_forest_model_bin.joblib"
model_mult_path = "D:/Area de trabalho/Faculdade/BioData/Q-Cyber/QML/IA/weights/traditional/random_forest_model_mult.joblib"

model_bin = joblib.load(model_bin_path)
model_mult = joblib.load(model_mult_path)

In [2]:
df = pd.read_csv("D:/Area de trabalho/Faculdade/BioData/Q-Cyber/QML/data/dados_de_teste.csv").sample(10)

gt_bin_labels = df["Attack_label"].tolist()
gt_mult_labels = df["Attack_type"].tolist()

x_test = df.drop(["Attack_label", "Attack_type"], axis=1)
csv_str_bin = x_test.reset_index(drop=True)
dict_bin = x_test.to_dict(orient='records')

In [3]:
x_test

,tcp.ack,tcp.ack_raw,tcp.checksum,tcp.dstport,tcp.flags,tcp.flags.ack,tcp.seq
2166,1.000000e+00,1.013594e+09,27534.0,4321.0,24.0,1.0,454049.0
6900,1.878000e+03,1.677904e+09,61092.0,80.0,16.0,1.0,87.0
26492,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0
28420,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0
29189,6.342370e+05,2.372259e+09,44121.0,60944.0,16.0,1.0,177787035.0
22177,2.010842e+09,2.010842e+09,16032.0,80.0,2.0,0.0,0.0
12271,0.000000e+00,0.000000e+00,24686.0,80.0,2.0,0.0,0.0
31012,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0
20212,1.689850e+05,2.371794e+09,15011.0,60944.0,24.0,1.0,42959576.0
11633,5.900000e+01,3.140697e+09,33649.0,63158.0,16.0,1.0,6.0


Usando o modelo binário

In [4]:
results = model_bin.predict(x_test)

In [5]:
print(f"Resultados: {results}")
print(f"Ground Truth: {gt_bin_labels}")

Resultados: [1 1 1 1 0 1 1 1 0 0]
Ground Truth: [1, 1, 1, 1, 0, 1, 1, 1, 0, 0]


Usando o modelo multiclasse

In [6]:
attack_index = np.where(results == 1)[0]
attack_index

array([0, 1, 2, 3, 5, 6, 7])

In [7]:
# Passando somente o que é ataque para o modelo multiclasse
x_test = x_test.reset_index(drop=True)
x_test_mult = x_test.loc[attack_index]

# Filtrando somente as amostras que foram classificadas como ataque
gt_mult_labels_filtered = [gt_mult_labels[i] for i in attack_index]

print(f"Original ground truth labels: {gt_mult_labels}")
print(f"Filtered ground truth labels: {gt_mult_labels_filtered}")

Original ground truth labels: [10, 8, 4, 4, 7, 9, 12, 2, 7, 7]
Filtered ground truth labels: [10, 8, 4, 4, 9, 12, 2]


In [8]:
results = model_mult.predict(x_test_mult)

In [10]:
print(f"Resultados: {results}")
print(f"Ground Truth: {gt_mult_labels_filtered}")

Resultados: [9 7 4 4 8 1 4]
Ground Truth: [10, 8, 4, 4, 9, 12, 2]


In [16]:
# Decoding the filtered labels

label_map = {
    'Backdoor': 0,
    'DDoS_HTTP': 1,
    'DDoS_ICMP': 2,
    'DDoS_TCP': 3,
    'DDoS_UDP': 4,
    'Fingerprinting': 5,
    'MITM': 6,
    'Password': 7,
    'Port_Scanning': 8,
    'Ransomware': 9,
    'SQL_injection': 10,
    'Uploading': 11,
    'Vulnerability_scanner': 12,
    'XSS': 13
}

In [17]:
def decode_type_attack(array_list):
    inv_label_map = {v: k for k, v in label_map.items()}
    decoded_labels = [inv_label_map[label] for label in array_list]
    print(f"Decoded Ground Truth: {decoded_labels}")

In [18]:
decode_type_attack(gt_mult_labels_filtered)

Decoded Ground Truth: ['SQL_injection', 'Port_Scanning', 'DDoS_UDP', 'DDoS_UDP', 'Ransomware', 'Vulnerability_scanner', 'DDoS_ICMP']


In [19]:
decode_type_attack(results)

Decoded Ground Truth: ['Ransomware', 'Password', 'DDoS_UDP', 'DDoS_UDP', 'Port_Scanning', 'DDoS_HTTP', 'DDoS_UDP']
